In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-14B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import pandas as pd
original_df = pd.read_csv(paths.PREDICTIONS / "Qwen_14B_predictions_Trainee.csv")
original_df

,QA_ID,Origin,data_source,Raw_Response,Physician_Removed,Original,answer_corr,data_source.1
0,Merge Q1300,ID0002,NaN,Given the patient's progressive ptosis over 9 ...,D,D,D,jama
1,Merge Q2,ID0003,NaN,The patient's symptoms and findings suggest a ...,D,D,G,medxpert
2,Merge Q2,ID0007,NaN,The patient's symptoms include persistent weak...,A,A,B,medbullets
3,Merge Q2,ID0009,NaN,Answer: D\n\nThe patient's symptoms and physic...,D,D,D,jama
4,Merge Q5,ID0010,NaN,The correct method for transporting an amputat...,H,H,H,medxpert
...,...,...,...,...,...,...,...,...
1295,Merge Q1295,ID1995,NaN,The patient exhibits symptoms consistent with ...,D,D,D,mmlu
1296,Merge Q1297,ID1996,NaN,"Given the patient's presentation of fever, hea...",D,D,D,mmlu
1297,Merge Q1297,ID1997,NaN,The patient's symptoms suggest cyclical mastal...,C,C,B,medxpert
1298,Merge Q1299,ID1998,NaN,The question asks about which structure would ...,A,D,A,medbullets


In [2]:
# Calculate spurious rate function
def calculate_spurious_rate(df, original_col, comparison_col, correct_answer_col):
    """
    Calculate the percentage of questions that were:
    1. Answered CORRECTLY in the original_col (Original == answer_corr)
    2. Answered INCORRECTLY in the comparison_col (comparison_col != answer_corr)
    
    Spurious rate = (count of [Original correct AND comparison incorrect]) / (count of [Original correct])
    """
    # Step 1: Filter rows where Original is correct and has valid data
    original_correct_mask = (df[original_col].notna()) & (df[original_col] == df[correct_answer_col])
    original_correct = df[original_correct_mask].copy()
    
    total_original_correct = len(original_correct)
    
    if total_original_correct == 0:
        return 0.0, 0, 0
    
    # Step 2: Among those where Original was correct, find where comparison is incorrect
    comparison_incorrect_mask = (
        original_correct[comparison_col].notna() & 
        (original_correct[comparison_col] != original_correct[correct_answer_col])
    )
    
    spurious_count = comparison_incorrect_mask.sum()
    spurious_rate = (spurious_count / total_original_correct) * 100
    
    return spurious_rate, spurious_count, total_original_correct

# Overall spurious rate for Physician_Removed
print("="*80)
print("PHYSICIAN REMOVED - OVERALL SPURIOUS RATE")
print("="*80)
print(f"{'Model':<30} {'Spurious Rate (%)':<20} {'Count':<15} {'Total Correct in Original'}")
print("-"*80)

overall_rate, overall_count, overall_total = calculate_spurious_rate(
    original_df, 
    'Original',
    'Physician_Removed', 
    'answer_corr'
)

print(f"{'Physician_Removed':<30} {overall_rate:>18.2f}% {overall_count:>14} / {overall_total}")
print("\n")

# Breakdown by data source
print("="*80)
print("PHYSICIAN REMOVED - SPURIOUS RATES BY DATA SOURCE")
print("="*80)

data_sources = original_df['data_source.1'].dropna().unique()
breakdown_results = {}

print(f"{'Data Source':<20} {'Spurious Rate (%)':<20} {'Count':<15} {'Total Correct in Original'}")
print("-"*80)

for source in sorted(data_sources):
    source_df = original_df[original_df['data_source.1'] == source]
    
    rate, count, total = calculate_spurious_rate(
        source_df,
        'Original',
        'Physician_Removed',
        'answer_corr'
    )
    
    breakdown_results[source] = {
        'rate': rate,
        'count': count,
        'total': total
    }
    
    print(f"{source:<20} {rate:>18.2f}% {count:>14} / {total}")

# Create summary DataFrame
summary_data = [{
    'Model': 'Physician_Removed',
    'Data Source': 'Overall',
    'Spurious Rate (%)': overall_rate,
    'Spurious Count': overall_count,
    'Total Original Correct': overall_total
}]

for source, stats in breakdown_results.items():
    summary_data.append({
        'Model': 'Physician_Removed',
        'Data Source': source,
        'Spurious Rate (%)': stats['rate'],
        'Spurious Count': stats['count'],
        'Total Original Correct': stats['total']
    })

summary_df = pd.DataFrame(summary_data)

print("\n")
print("="*80)
print("SUMMARY TABLE - PHYSICIAN REMOVED")
print("="*80)
print(summary_df.to_string(index=False))

# Save results
summary_df.to_csv(paths.TABLES / "physician_removed_spurious_rate_analysis.csv", index=False)

print("\n\nResults saved to paths.TABLES / "physician_removed_spurious_rate_analysis.csv"")

# Additional diagnostic info
print("\n")
print("="*80)
print("DIAGNOSTIC INFO - PHYSICIAN REMOVED")
print("="*80)
print(f"Total rows in dataset: {len(original_df)}")
print(f"Rows with non-null 'Original': {original_df['Original'].notna().sum()}")
print(f"Rows with non-null 'Physician_Removed': {original_df['Physician_Removed'].notna().sum()}")
print(f"Rows where 'Original' is correct: {(original_df['Original'] == original_df['answer_corr']).sum()}")
print(f"Rows where 'Physician_Removed' is correct: {(original_df['Physician_Removed'] == original_df['answer_corr']).sum()}")

# Show data source distribution
print("\n")
print("Data source distribution:")
print(original_df['data_source.1'].value_counts())

# Additional analysis: Show examples of spurious errors
print("\n")
print("="*80)
print("EXAMPLES OF SPURIOUS ERRORS (First 10)")
print("="*80)
spurious_mask = (
    (original_df['Original'].notna()) & 
    (original_df['Original'] == original_df['answer_corr']) &
    (original_df['Physician_Removed'].notna()) &
    (original_df['Physician_Removed'] != original_df['answer_corr'])
)

spurious_examples = original_df[spurious_mask][['QA_ID', 'Origin', 'data_source.1', 'Original', 'Physician_Removed', 'answer_corr']].head(10)
print(spurious_examples.to_string(index=False))

print(f"\nTotal spurious errors found: {spurious_mask.sum()}")

# Create a detailed spurious errors file
spurious_errors_df = original_df[spurious_mask][['QA_ID', 'Origin', 'data_source', 'Raw_Response', 'Original', 'Physician_Removed', 'answer_corr']]
spurious_errors_df.to_csv(paths.TABLES / "physician_removed_spurious_errors_detailed.csv", index=False)
print(f"Detailed spurious errors saved to paths.TABLES / "physician_removed_spurious_errors_detailed.csv" ({len(spurious_errors_df)} rows)")

PHYSICIAN REMOVED - OVERALL SPURIOUS RATE
Model                          Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
Physician_Removed                           15.30%            112 / 732


PHYSICIAN REMOVED - SPURIOUS RATES BY DATA SOURCE
Data Source          Spurious Rate (%)    Count           Total Correct in Original
--------------------------------------------------------------------------------
jama                              18.13%             68 / 375
medbullets                         7.50%              9 / 120
medxpert                          27.54%             19 / 69
mmlu                               9.52%             16 / 168


SUMMARY TABLE - PHYSICIAN REMOVED
             Model Data Source  Spurious Rate (%)  Spurious Count  Total Original Correct
 Physician_Removed     Overall          15.300546             112                     732
 Physician_Removed        jama 